# E1 — Данные и сценарии (Ростов-на-Дону)

Leakage-free сплит по годам: **train = 2018, 2019**, **in-distribution test = 2020**, **OOD = 2021–2023** (OOD задействуется в E5). Сбор траекторий rule-based с возбуждением (шум + **PRBS** на актуаторы) для идентифицируемости; кривая бюджета данных {1,3,…} сут и число обусловленности κ. Перед env вызывается `apply_rostov_soil()` (зашито в `_make_env`).

In [1]:
import os, sys, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath("."))
import article_experiment_utils as U
import protocol_config as P

# FAST_MODE smoke (tiny data) vs article-grade. Toggle via env var ARTICLE_FAST=0.
FAST_MODE = os.environ.get("ARTICLE_FAST", "1") == "1"
pc = P.DEFAULT.resolved(FAST_MODE)
RES = U.results_dir()
ECON = P.read_env_economics(pc.location)
CORR, PRICES = ECON["corridors"], ECON["prices"]
print("FAST_MODE", FAST_MODE, "| seeds", tuple(pc.seeds), "| budgets", pc.budgets_days,
      "| n_days_train/test", pc.n_days_train, pc.n_days_test)

FAST_MODE False | seeds (0, 1, 2, 3, 4, 5, 6, 7, 8, 9) | budgets (1, 3, 7, 14, 30, 60) | n_days_train/test 60 60


## Таблица сплитов

In [2]:
splits = pc.train_scenarios() + [pc.test_scenario()] + pc.ood_scenarios()
splits_df = pd.DataFrame(splits)
U.save_table(splits_df, RES / "tables" / "e1_splits.csv"); splits_df

,year,start_date,n_days,role
0,2018,2018-03-01,60,train
1,2019,2019-03-01,60,train
2,2020,2020-03-01,60,test_in_dist
3,2021,2021-03-01,60,ood
4,2022,2022-03-01,60,ood
5,2023,2023-03-01,60,ood


## Сбор обучающих датасетов (по годам, rule-based + шум + PRBS) и тест-сезона

In [3]:
train_parts = []
for sc in pc.train_scenarios():
    cfg = pc.cfg_for(sc, seed=0)
    d = U.collect_rule_based_dataset(cfg, n_days=pc.n_days_train, prbs_scale=0.3)
    U.save_dataset(d, RES / "datasets" / f"e1_train_{sc['year']}.npz")
    train_parts.append(d)
    print("train", sc["year"], "rows", len(d.states), "src", d.meta["source"])
train = U.aggregate_trajectories(train_parts, pc.base_cfg(pc.n_days_train))
tcfg = pc.cfg_for(pc.test_scenario(), seed=1)
test = U.collect_rule_based_dataset(tcfg, n_days=pc.n_days_test, prbs_scale=0.0)
U.save_dataset(test, RES / "datasets" / "e1_test_2020.npz")
print("aggregated train rows", len(train.states), "| test rows", len(test.states))

train 2018 rows 5760 src rule_based+prbs


train 2019 rows 5760 src rule_based+prbs


aggregated train rows 11520 | test rows 5760


## Кривая бюджета данных: κ и разреженность по бюджетам

In [4]:
rows = []
for b in pc.budgets_days:
    sub = train.subset_steps(b * pc.steps_per_day)
    bundle = U.fit_sindy(sub, feature_variant="physics", library_degree=1, period=float(pc.period))
    rows.append({"budget_days": b, "rows": len(sub.states), "kappa": bundle.condition_number,
                 "nonzero": int(np.count_nonzero(bundle.model.coefficients()))})
budget_df = pd.DataFrame(rows)
U.save_table(budget_df, RES / "tables" / "e1_budget_kappa.csv"); budget_df

,budget_days,rows,kappa,nonzero
0,1,96,94.944547,57
1,3,288,59.928092,34
2,7,672,49.660299,31
3,14,1344,39.489062,30
4,30,2880,40.509575,31
5,60,5760,55.679482,33


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(train.to_frame()["time_h"]/24.0, train.states[:, 0]); axes[0].set_title("train: T_in"); axes[0].set_xlabel("сутки"); axes[0].grid(alpha=.3)
axes[1].plot(budget_df["budget_days"], budget_df["kappa"], marker="o"); axes[1].set_title("κ vs бюджет данных"); axes[1].set_xlabel("сут"); axes[1].set_ylabel("κ"); axes[1].grid(alpha=.3)
U.save_figure(fig, RES / "figures" / "e1_dataset_and_kappa.png"); plt.close(fig); print("saved e1 figure")

saved e1 figure


**Итог E1.** Сформированы train/test датасеты на Ростове с возбуждением, сохранены `.npz`; получена кривая κ по бюджету данных.